# 05_Main_Model - Stronger Dual-View SER

Notebook này không xóa bản `05_Dual_View_Fusion_SER.ipynb` cũ. Mục tiêu là thử lại mô hình đề xuất theo hướng mạnh hơn:

- **1D MFCC branch** dùng residual/dilated Conv1D + attention pooling.
- **2D log-Mel branch** dùng CNN sâu hơn + channel attention cho spectrogram.
- **Gated fusion** trộn hai embedding thay vì nối thẳng đặc trưng.
- **Optional BiGRU** để học ngữ cảnh dài hơn theo hướng CNN-GRU/LSTM trong literature.
- **Optional 2D-only log-Mel baseline** để kiểm tra xem nhánh 2D có thực sự mạnh hơn dual fusion không.

Kết quả cũ của dual-view yếu hơn 1D-CNN chủ yếu vì: 2D branch còn mỏng, fusion tăng tham số nhưng chưa tăng tín hiệu đủ mạnh, augmentation chưa được ablation rõ, và strict speaker-aware khó hơn nhiều so với paper random/single-dataset. Notebook này sửa theo hướng có kiểm soát, vẫn chạy được trên Kaggle/Colab.


## References Used for Design

1. **Ahmed et al. - Ensemble 1D-CNN-LSTM-GRU**: dùng augmentation, local feature acquiring blocks, GRU/LSTM branch và weighted ensemble. Ý tưởng lấy về: thêm recurrent variant và ensemble xác suất.
2. **Lee & Nadeem - MFCC 1D-CNN with Channel/Spatial Attention**: dùng attention trên MFCC augmented. Ý tưởng lấy về: attention pooling/channel attention để model tập trung vào vùng đặc trưng cảm xúc.
3. **DCRF-BiLSTM feature engineering paper**: dùng nhiều feature, augmentation, BiLSTM và structured prediction. Ý tưởng lấy về: feature diversity + temporal model + phân tích protocol.

Lưu ý: nhiều paper báo accuracy rất cao theo từng dataset riêng hoặc split không speaker-aware. Notebook này vẫn giữ `combined_random`, `combined_strict`, và `single_dataset` để so sánh công bằng hơn.


## 1. Imports and Reproducibility


In [ ]:
from pathlib import Path
import os, json, time, random, shutil, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa, soundfile as sf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
try:
    from IPython.display import display
except Exception:
    def display(x): print(x)

SEED = int(os.getenv('SEED', '42'))
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
DISABLE_CUDNN_RNN = os.getenv('DISABLE_CUDNN_RNN', '1') == '1'
if DISABLE_CUDNN_RNN:
    torch.backends.cudnn.enabled = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE, '| cudnn:', torch.backends.cudnn.enabled)


## 2. Path Setup


In [ ]:
def find_ser_processed():
    candidates = []
    env_path = os.getenv('SER_PROCESSED', '').strip()
    if env_path:
        candidates.append(Path(env_path))
    candidates.extend([
        Path('/kaggle/input/datasets/quanghuy225/ser-processed/ser_processed'),
        Path('/kaggle/input/ser-processed/ser_processed'),
        Path('/kaggle/input/ser-processed'),
        Path('/kaggle/working/ser_processed'),
        Path.cwd() / 'ser_processed',
        Path.cwd().parent / 'ser_processed',
        Path.cwd().parent / '01&02_Data_and_DataProcessing' / 'ser_processed',
        Path('D:/UTE/Speech Programming/Speech Project/01&02_Data_and_DataProcessing/ser_processed'),
    ])
    for candidate in candidates:
        if (candidate / 'metadata.csv').exists():
            return candidate.resolve()
    for root in [Path('/kaggle/input'), Path.cwd(), Path.cwd().parent]:
        if root.exists():
            for metadata_path in root.rglob('metadata.csv'):
                if metadata_path.parent.name == 'ser_processed':
                    return metadata_path.parent.resolve()
    raise FileNotFoundError('Cannot find ser_processed/metadata.csv')

SER_PROCESSED = find_ser_processed()
AUDIO_16K_DIR = SER_PROCESSED / 'audio_16k'
PROJECT_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / '05_Dual_View_Stronger_outputs'
REPORT_DIR = OUTPUT_DIR / 'reports'
FIGURE_DIR = OUTPUT_DIR / 'figures'
MODEL_DIR = OUTPUT_DIR / 'models'
PRED_DIR = OUTPUT_DIR / 'predictions'
CACHE_DIR = OUTPUT_DIR / 'cache'
for d in [REPORT_DIR, FIGURE_DIR, MODEL_DIR, PRED_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('SER_PROCESSED:', SER_PROCESSED)
print('OUTPUT_DIR:', OUTPUT_DIR)


## 3. Configuration


In [ ]:
QUICK_RUN = os.getenv('QUICK_RUN', '0') == '1'
QUICK_RUN_PER_GROUP = int(os.getenv('QUICK_RUN_PER_GROUP', '24'))

TARGET_SR = 16_000
TARGET_DURATION = float(os.getenv('TARGET_DURATION', '3.0'))
TARGET_LENGTH = int(TARGET_SR * TARGET_DURATION)
N_FFT = 400
WIN_LENGTH = 400
HOP_LENGTH = 160
N_MFCC = int(os.getenv('N_MFCC', '40'))
N_MELS = int(os.getenv('N_MELS', '96'))
MAX_FRAMES = int(1 + TARGET_LENGTH // HOP_LENGTH)

COMMON_EMOTIONS = ['neutral', 'happy', 'sad', 'angry', 'fear', 'disgust']
LABEL_TO_ID = {label: i for i, label in enumerate(COMMON_EMOTIONS)}
ID_TO_LABEL = {i: label for label, i in LABEL_TO_ID.items()}

BATCH_SIZE = int(os.getenv('BATCH_SIZE', '64'))
MAX_EPOCHS = int(os.getenv('MAX_EPOCHS', '45'))
PATIENCE = int(os.getenv('PATIENCE', '10'))
LR = float(os.getenv('LR', '2e-3'))
WEIGHT_DECAY = float(os.getenv('WEIGHT_DECAY', '1e-4'))
DROPOUT = float(os.getenv('DROPOUT', '0.35'))
LABEL_SMOOTHING = float(os.getenv('LABEL_SMOOTHING', '0.04'))

MODEL_VARIANTS = [x.strip() for x in os.getenv('MODEL_VARIANTS', 'dual_attention,dual_bigru,logmel_2d').split(',') if x.strip()]
RUN_COMBINED_RANDOM = os.getenv('RUN_COMBINED_RANDOM', '1') == '1'
RUN_COMBINED_STRICT = os.getenv('RUN_COMBINED_STRICT', '1') == '1'
RUN_SINGLE_DATASET = os.getenv('RUN_SINGLE_DATASET', '1') == '1'
CACHE_FEATURES = os.getenv('CACHE_FEATURES', '1') == '1'
USE_CLASS_WEIGHTS = os.getenv('USE_CLASS_WEIGHTS', '1') == '1'
USE_AUGMENTATION = os.getenv('USE_AUGMENTATION', '1') == '1'

print({
    'QUICK_RUN': QUICK_RUN, 'MODEL_VARIANTS': MODEL_VARIANTS,
    'MAX_EPOCHS': MAX_EPOCHS, 'RUN_SINGLE_DATASET': RUN_SINGLE_DATASET,
    'USE_AUGMENTATION': USE_AUGMENTATION
})


## 4. Load Metadata and Build Splits


In [ ]:
metadata = pd.read_csv(SER_PROCESSED / 'metadata.csv')
metadata = metadata[metadata['emotion'].isin(COMMON_EMOTIONS)].copy()
metadata = metadata[metadata.get('readable', True) == True].copy()
metadata['label_id'] = metadata['emotion'].map(LABEL_TO_ID).astype(int)
metadata['speaker_id'] = metadata['speaker_id'].astype(str)

if QUICK_RUN:
    metadata = (metadata
        .sort_values(['dataset', 'emotion', 'sample_id'])
        .groupby(['dataset', 'emotion'], group_keys=False)
        .head(QUICK_RUN_PER_GROUP)
        .reset_index(drop=True))
else:
    metadata = metadata.reset_index(drop=True)

metadata['split_strict_original'] = metadata['split'].astype(str).str.lower() if 'split' in metadata.columns else 'train'
metadata.loc[~metadata['split_strict_original'].isin(['train', 'validation', 'test']), 'split_strict_original'] = 'train'

def apply_project_strict_split(df):
    df = df.copy()
    df['strict_include'] = df['dataset'] != 'TESS'
    df['split_strict_project'] = df['split_strict_original']
    savee_plan = {'savee_DC': 'train', 'savee_JE': 'train', 'savee_JK': 'validation', 'savee_KL': 'test'}
    savee_mask = df['dataset'].eq('SAVEE')
    df.loc[savee_mask, 'split_strict_project'] = df.loc[savee_mask, 'speaker_id'].map(savee_plan).fillna('train')
    df.loc[df['dataset'].eq('TESS'), 'split_strict_project'] = 'excluded'
    return df

metadata = apply_project_strict_split(metadata)
train_idx, temp_idx = train_test_split(metadata.index, test_size=0.30, random_state=SEED, stratify=metadata['label_id'])
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=SEED, stratify=metadata.loc[temp_idx, 'label_id'])
metadata['split_random'] = 'train'
metadata.loc[val_idx, 'split_random'] = 'validation'
metadata.loc[test_idx, 'split_random'] = 'test'

print('Metadata shape:', metadata.shape)
display(metadata.groupby(['dataset', 'emotion']).size().unstack(fill_value=0))
print('\nRandom split counts:', metadata['split_random'].value_counts().to_dict())
print('Strict project split counts:', metadata['split_strict_project'].value_counts().to_dict())
display(metadata.groupby(['dataset', 'split_strict_project']).size().unstack(fill_value=0))


## 5. Feature Extraction: MFCC Sequence + log-Mel Image


In [ ]:
def center_crop_or_pad(y, target_length=TARGET_LENGTH):
    if len(y) > target_length:
        start = (len(y) - target_length) // 2
        y = y[start:start + target_length]
    elif len(y) < target_length:
        pad = target_length - len(y)
        y = np.pad(y, (pad // 2, pad - pad // 2), mode='constant')
    return y.astype(np.float32)

def read_audio(row):
    cached = AUDIO_16K_DIR / f'{row.sample_id}.wav'
    if cached.exists():
        y, sr = sf.read(cached)
        if y.ndim > 1:
            y = y.mean(axis=1)
        if sr != TARGET_SR:
            y = librosa.resample(y.astype(np.float32), orig_sr=sr, target_sr=TARGET_SR)
    else:
        y, sr = librosa.load(row.filepath, sr=TARGET_SR, mono=True)
    y = center_crop_or_pad(y, TARGET_LENGTH)
    peak = np.max(np.abs(y)) + 1e-8
    if peak > 1.0:
        y = y / peak
    return y

def pad_time(x, target, axis=-1):
    if x.shape[axis] > target:
        sl = [slice(None)] * x.ndim
        sl[axis] = slice(0, target)
        return x[tuple(sl)]
    if x.shape[axis] < target:
        pad_width = [(0, 0)] * x.ndim
        pad_width[axis] = (0, target - x.shape[axis])
        return np.pad(x, pad_width, mode='constant')
    return x

def extract_dual_features(row):
    y = read_audio(row)
    mfcc = librosa.feature.mfcc(y=y, sr=TARGET_SR, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH)
    delta = librosa.feature.delta(mfcc)
    mfcc_feat = np.concatenate([mfcc, delta], axis=0)
    mfcc_feat = pad_time(mfcc_feat, MAX_FRAMES, axis=1).T.astype(np.float32)

    mel = librosa.feature.melspectrogram(y=y, sr=TARGET_SR, n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH, n_mels=N_MELS, power=2.0)
    logmel = librosa.power_to_db(mel, ref=np.max)
    logmel = pad_time(logmel, MAX_FRAMES, axis=1).astype(np.float32)
    return mfcc_feat, logmel

cache_name = f'stronger_dual_mfcc{N_MFCC}_mel{N_MELS}_{int(TARGET_DURATION)}s_{TARGET_SR}hz_{len(metadata)}files.npz'
cache_path = CACHE_DIR / cache_name
if CACHE_FEATURES and cache_path.exists():
    data = np.load(cache_path)
    X_mfcc = data['X_mfcc']; X_mel = data['X_mel']; y = data['y']
    print('Loaded cache:', cache_path)
else:
    X_mfcc, X_mel = [], []
    start = time.time()
    for i, row in enumerate(metadata.itertuples(index=False)):
        a, b = extract_dual_features(row)
        X_mfcc.append(a); X_mel.append(b)
        if (i + 1) % 500 == 0:
            print(f'Extracted {i+1}/{len(metadata)} files in {(time.time()-start)/60:.1f} min')
    X_mfcc = np.stack(X_mfcc)
    X_mel = np.stack(X_mel)
    y = metadata['label_id'].to_numpy(np.int64)
    if CACHE_FEATURES:
        np.savez_compressed(cache_path, X_mfcc=X_mfcc, X_mel=X_mel, y=y)
        print('Saved cache:', cache_path)

MFCC_DIM = X_mfcc.shape[-1]
print('X_mfcc:', X_mfcc.shape, 'X_mel:', X_mel.shape, 'y:', y.shape)


## 6. Dataset, Scaling and Train-Only Augmentation


In [ ]:
def compute_scalers(train_idx):
    mfcc_mean = X_mfcc[train_idx].mean(axis=(0, 1), keepdims=True)
    mfcc_std = X_mfcc[train_idx].std(axis=(0, 1), keepdims=True) + 1e-6
    mel_mean = X_mel[train_idx].mean()
    mel_std = X_mel[train_idx].std() + 1e-6
    return mfcc_mean.astype(np.float32), mfcc_std.astype(np.float32), float(mel_mean), float(mel_std)

def augment_pair(mfcc, mel):
    if not USE_AUGMENTATION:
        return mfcc, mel
    if random.random() < 0.35:
        shift = random.randint(-16, 16)
        mfcc = np.roll(mfcc, shift, axis=0)
        mel = np.roll(mel, shift, axis=1)
    if random.random() < 0.35:
        mfcc = mfcc + np.random.normal(0, 0.02, mfcc.shape).astype(np.float32)
        mel = mel + np.random.normal(0, 0.02, mel.shape).astype(np.float32)
    if random.random() < 0.45:
        width = random.randint(8, 32)
        start = random.randint(0, max(0, mfcc.shape[0] - width))
        mfcc[start:start+width, :] = 0
        mel[:, start:start+width] = 0
    if random.random() < 0.45:
        width = random.randint(6, 18)
        start = random.randint(0, max(0, mel.shape[0] - width))
        mel[start:start+width, :] = 0
    return mfcc, mel

class DualFeatureDataset(Dataset):
    def __init__(self, indices, scalers, train=False):
        self.indices = np.array(indices)
        self.mfcc_mean, self.mfcc_std, self.mel_mean, self.mel_std = scalers
        self.train = train
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        i = self.indices[idx]
        mfcc = ((X_mfcc[i] - self.mfcc_mean.squeeze(0)) / self.mfcc_std.squeeze(0)).astype(np.float32)
        mel = ((X_mel[i] - self.mel_mean) / self.mel_std).astype(np.float32)
        if self.train:
            mfcc, mel = augment_pair(mfcc.copy(), mel.copy())
        return torch.from_numpy(mfcc.T), torch.from_numpy(mel[None, :, :]), torch.tensor(y[i], dtype=torch.long)

def make_loaders(split_map):
    train_idx, val_idx, test_idx = split_map['train'], split_map['validation'], split_map['test']
    scalers = compute_scalers(train_idx)
    loaders = {
        'train': DataLoader(DualFeatureDataset(train_idx, scalers, train=True), batch_size=BATCH_SIZE, shuffle=True, num_workers=0),
        'validation': DataLoader(DualFeatureDataset(val_idx, scalers, train=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=0),
        'test': DataLoader(DualFeatureDataset(test_idx, scalers, train=False), batch_size=BATCH_SIZE, shuffle=False, num_workers=0),
    }
    return loaders, scalers


## 7. Stronger Model Architectures


In [ ]:
class SE1D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(8, channels // reduction)
        self.net = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(channels, hidden), nn.ReLU(), nn.Linear(hidden, channels), nn.Sigmoid())
    def forward(self, x):
        return x * self.net(x).unsqueeze(-1)

class SE2D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(8, channels // reduction)
        self.net = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(channels, hidden), nn.ReLU(), nn.Linear(hidden, channels), nn.Sigmoid())
    def forward(self, x):
        return x * self.net(x).view(x.size(0), x.size(1), 1, 1)

class AttentionPool1D(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.attn = nn.Sequential(nn.Conv1d(channels, channels // 2, 1), nn.Tanh(), nn.Conv1d(channels // 2, 1, 1))
    def forward(self, x):
        w = torch.softmax(self.attn(x), dim=-1)
        return (x * w).sum(dim=-1)

class ResConv1DBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=5, dilation=1, dropout=DROPOUT):
        super().__init__()
        pad = dilation * (kernel // 2)
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel, padding=pad, dilation=dilation),
            nn.BatchNorm1d(out_ch), nn.GELU(), nn.Dropout(dropout),
            nn.Conv1d(out_ch, out_ch, kernel, padding=pad, dilation=dilation),
            nn.BatchNorm1d(out_ch), nn.GELU()
        )
        self.shortcut = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
        self.se = SE1D(out_ch)
        self.pool = nn.MaxPool1d(2)
    def forward(self, x):
        x = self.se(self.conv(x) + self.shortcut(x))
        return self.pool(x)

class MFCCAttentionBranch(nn.Module):
    def __init__(self, in_channels, embed_dim=256):
        super().__init__()
        self.blocks = nn.Sequential(
            ResConv1DBlock(in_channels, 96, dilation=1),
            ResConv1DBlock(96, 160, dilation=2),
            ResConv1DBlock(160, 224, dilation=3),
        )
        self.pool = AttentionPool1D(224)
        self.proj = nn.Sequential(nn.Linear(224, embed_dim), nn.LayerNorm(embed_dim), nn.GELU(), nn.Dropout(DROPOUT))
    def forward(self, x):
        seq = self.blocks(x)
        emb = self.proj(self.pool(seq))
        return emb, seq.transpose(1, 2)

class MelCNNBranch(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()
        def block(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.GELU(),
                                 nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.GELU(),
                                 SE2D(o), nn.MaxPool2d(2), nn.Dropout2d(0.10))
        self.cnn = nn.Sequential(block(1, 32), block(32, 64), block(64, 128), block(128, 192))
        self.proj = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(192, embed_dim), nn.LayerNorm(embed_dim), nn.GELU(), nn.Dropout(DROPOUT))
    def forward(self, x):
        return self.proj(self.cnn(x))

class DualAttentionFusionSER(nn.Module):
    def __init__(self, mfcc_dim, num_classes, embed_dim=256):
        super().__init__()
        self.mfcc = MFCCAttentionBranch(mfcc_dim, embed_dim)
        self.mel = MelCNNBranch(embed_dim)
        self.gate = nn.Sequential(nn.Linear(embed_dim * 2, embed_dim), nn.GELU(), nn.Linear(embed_dim, embed_dim), nn.Sigmoid())
        self.cls = nn.Sequential(nn.LayerNorm(embed_dim), nn.Dropout(DROPOUT), nn.Linear(embed_dim, 128), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(128, num_classes))
    def forward(self, mfcc, mel):
        a, _ = self.mfcc(mfcc)
        b = self.mel(mel)
        g = self.gate(torch.cat([a, b], dim=1))
        fused = g * a + (1 - g) * b
        return self.cls(fused)

class DualBiGRUFusionSER(nn.Module):
    def __init__(self, mfcc_dim, num_classes, embed_dim=256):
        super().__init__()
        self.mfcc = MFCCAttentionBranch(mfcc_dim, embed_dim)
        self.gru = nn.GRU(224, 128, batch_first=True, bidirectional=True)
        self.gru_pool = nn.Sequential(nn.Linear(256, embed_dim), nn.LayerNorm(embed_dim), nn.GELU(), nn.Dropout(DROPOUT))
        self.mel = MelCNNBranch(embed_dim)
        self.gate = nn.Sequential(nn.Linear(embed_dim * 2, embed_dim), nn.GELU(), nn.Linear(embed_dim, embed_dim), nn.Sigmoid())
        self.cls = nn.Sequential(nn.LayerNorm(embed_dim), nn.Dropout(DROPOUT), nn.Linear(embed_dim, 128), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(128, num_classes))
    def forward(self, mfcc, mel):
        _, seq = self.mfcc(mfcc)
        if DISABLE_CUDNN_RNN and seq.is_cuda:
            seq = seq.contiguous()
        out, _ = self.gru(seq)
        a = self.gru_pool(out.mean(dim=1))
        b = self.mel(mel)
        g = self.gate(torch.cat([a, b], dim=1))
        return self.cls(g * a + (1 - g) * b)

class LogMel2DCNN(nn.Module):
    def __init__(self, mfcc_dim, num_classes, embed_dim=256):
        super().__init__()
        self.mel = MelCNNBranch(embed_dim)
        self.cls = nn.Sequential(nn.LayerNorm(embed_dim), nn.Dropout(DROPOUT), nn.Linear(embed_dim, 128), nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(128, num_classes))
    def forward(self, mfcc, mel):
        return self.cls(self.mel(mel))

def build_model(variant):
    if variant == 'dual_attention':
        return DualAttentionFusionSER(MFCC_DIM, len(COMMON_EMOTIONS)).to(DEVICE)
    if variant == 'dual_bigru':
        return DualBiGRUFusionSER(MFCC_DIM, len(COMMON_EMOTIONS)).to(DEVICE)
    if variant == 'logmel_2d':
        return LogMel2DCNN(MFCC_DIM, len(COMMON_EMOTIONS)).to(DEVICE)
    raise ValueError(f'Unknown variant: {variant}')

for variant in MODEL_VARIANTS:
    m = build_model(variant)
    print(variant, 'parameters:', sum(p.numel() for p in m.parameters()))
    del m
torch.cuda.empty_cache()


## 8. Training and Evaluation Utilities


In [ ]:
def class_weights_for(indices):
    counts = np.bincount(y[indices], minlength=len(COMMON_EMOTIONS)).astype(np.float32)
    weights = counts.sum() / (len(COMMON_EMOTIONS) * np.maximum(counts, 1))
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, all_y, all_pred, all_prob = 0.0, [], [], []
    for mfcc, mel, target in loader:
        mfcc, mel, target = mfcc.to(DEVICE), mel.to(DEVICE), target.to(DEVICE)
        if is_train:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_train):
            logits = model(mfcc, mel)
            loss = criterion(logits, target)
            if is_train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()
        prob = torch.softmax(logits.detach(), dim=1)
        total_loss += loss.item() * len(target)
        all_y.extend(target.detach().cpu().numpy())
        all_pred.extend(prob.argmax(dim=1).cpu().numpy())
        all_prob.extend(prob.cpu().numpy())
    all_y = np.array(all_y); all_pred = np.array(all_pred); all_prob = np.array(all_prob)
    return {
        'loss': total_loss / max(1, len(all_y)),
        'accuracy': accuracy_score(all_y, all_pred),
        'macro_f1': f1_score(all_y, all_pred, average='macro', zero_division=0),
        'weighted_f1': f1_score(all_y, all_pred, average='weighted', zero_division=0),
        'macro_precision': precision_score(all_y, all_pred, average='macro', zero_division=0),
        'macro_recall': recall_score(all_y, all_pred, average='macro', zero_division=0),
        'y_true': all_y, 'y_pred': all_pred, 'prob': all_prob
    }

def plot_confusion(y_true, y_pred, title, path):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(COMMON_EMOTIONS))))
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=COMMON_EMOTIONS, yticklabels=COMMON_EMOTIONS)
    plt.title(title); plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout()
    plt.savefig(path, dpi=160); plt.close()

def train_one(run_name, variant, split_map):
    loaders, scalers = make_loaders(split_map)
    model = build_model(variant)
    weights = class_weights_for(split_map['train']) if USE_CLASS_WEIGHTS else None
    criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

    best_state, best_val, best_epoch, stale = None, -1, 0, 0
    history = []
    start = time.time()
    for epoch in range(1, MAX_EPOCHS + 1):
        tr = run_epoch(model, loaders['train'], criterion, optimizer)
        va = run_epoch(model, loaders['validation'], criterion, None)
        scheduler.step(va['macro_f1'])
        row = {'run_name': run_name, 'variant': variant, 'epoch': epoch,
               'train_loss': tr['loss'], 'train_macro_f1': tr['macro_f1'],
               'val_loss': va['loss'], 'val_macro_f1': va['macro_f1'], 'val_accuracy': va['accuracy'],
               'lr': optimizer.param_groups[0]['lr']}
        history.append(row)
        print(f"{run_name} | {variant} | epoch {epoch:02d} | train f1 {tr['macro_f1']:.4f} | val f1 {va['macro_f1']:.4f}")
        if va['macro_f1'] > best_val:
            best_val, best_epoch, stale = va['macro_f1'], epoch, 0
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= PATIENCE:
                print('Early stopping.')
                break

    model.load_state_dict(best_state)
    t0 = time.time()
    te = run_epoch(model, loaders['test'], criterion, None)
    inference_time = time.time() - t0
    train_time = time.time() - start

    torch.save({'model_state': best_state, 'variant': variant, 'run_name': run_name, 'best_epoch': best_epoch, 'config': {
        'N_MFCC': N_MFCC, 'N_MELS': N_MELS, 'TARGET_DURATION': TARGET_DURATION, 'COMMON_EMOTIONS': COMMON_EMOTIONS
    }}, MODEL_DIR / f'{run_name}_{variant}.pt')

    test_indices = split_map['test']
    pred_df = metadata.loc[test_indices, ['sample_id', 'dataset', 'speaker_id', 'emotion']].copy()
    pred_df['y_true'] = te['y_true']; pred_df['y_pred'] = te['y_pred']
    pred_df['pred_emotion'] = [ID_TO_LABEL[i] for i in te['y_pred']]
    for i, label in ID_TO_LABEL.items():
        pred_df[f'prob_{label}'] = te['prob'][:, i]
    pred_df.to_csv(PRED_DIR / f'predictions_{run_name}_{variant}.csv', index=False)
    plot_confusion(te['y_true'], te['y_pred'], f'{run_name} {variant}', FIGURE_DIR / f'confusion_matrix_{run_name}_{variant}.png')

    report = classification_report(te['y_true'], te['y_pred'], labels=list(range(len(COMMON_EMOTIONS))), target_names=COMMON_EMOTIONS, output_dict=True, zero_division=0)
    per_class = pd.DataFrame(report).T.reset_index().rename(columns={'index': 'class'})
    per_class.insert(0, 'variant', variant); per_class.insert(0, 'run_name', run_name)
    per_class.to_csv(REPORT_DIR / f'per_class_{run_name}_{variant}.csv', index=False)

    per_dataset = []
    for ds, part in pred_df.groupby('dataset'):
        per_dataset.append({
            'run_name': run_name, 'variant': variant, 'dataset': ds, 'n_samples': len(part),
            'accuracy': accuracy_score(part['y_true'], part['y_pred']),
            'macro_f1': f1_score(part['y_true'], part['y_pred'], average='macro', zero_division=0),
        })

    metric = {
        'run_name': run_name, 'variant': variant, 'split': 'test', 'n_samples': len(te['y_true']),
        'loss': te['loss'], 'accuracy': te['accuracy'], 'macro_f1': te['macro_f1'], 'weighted_f1': te['weighted_f1'],
        'macro_precision': te['macro_precision'], 'macro_recall': te['macro_recall'],
        'best_epoch': best_epoch, 'best_val_macro_f1': best_val,
        'train_time_sec': train_time, 'inference_time_sec': inference_time,
        'inference_ms_per_sample': 1000 * inference_time / max(1, len(te['y_true'])),
    }
    return metric, history, per_dataset


## 9. Experiment Definitions


In [ ]:
def split_from_column(col, include_mask=None):
    df = metadata if include_mask is None else metadata[include_mask]
    split_values = df[col].to_numpy()
    indices = df.index.to_numpy()
    out = {s: indices[split_values == s] for s in ['train', 'validation', 'test']}
    if len(out['validation']) == 0:
        train_sub, val_sub = train_test_split(out['train'], test_size=0.12, random_state=SEED, stratify=y[out['train']])
        out['train'], out['validation'] = train_sub, val_sub
    return out

experiments = []
if RUN_COMBINED_RANDOM:
    experiments.append(('combined_random', split_from_column('split_random')))
if RUN_COMBINED_STRICT:
    strict_mask = metadata['split_strict_project'].isin(['train', 'validation', 'test'])
    experiments.append(('combined_strict_no_tess', split_from_column('split_strict_project', strict_mask)))
if RUN_SINGLE_DATASET:
    for ds in sorted(metadata['dataset'].unique()):
        ds_df = metadata[metadata['dataset'].eq(ds)]
        idx_train, idx_temp = train_test_split(ds_df.index, test_size=0.30, random_state=SEED, stratify=metadata.loc[ds_df.index, 'label_id'])
        idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, random_state=SEED, stratify=metadata.loc[idx_temp, 'label_id'])
        experiments.append((f'single_{ds}', {'train': np.array(idx_train), 'validation': np.array(idx_val), 'test': np.array(idx_test)}))

for name, split_map in experiments:
    print(name, {k: len(v) for k, v in split_map.items()})


## 10. Run Training


In [ ]:
all_metrics, all_history, all_per_dataset = [], [], []
for exp_name, split_map in experiments:
    for variant in MODEL_VARIANTS:
        run_name = f'{exp_name}'
        metric, history, per_dataset = train_one(run_name, variant, split_map)
        all_metrics.append(metric)
        all_history.extend(history)
        all_per_dataset.extend(per_dataset)
        pd.DataFrame(all_metrics).to_csv(REPORT_DIR / 'stronger_dual_metrics.csv', index=False)
        pd.DataFrame(all_history).to_csv(REPORT_DIR / 'stronger_dual_training_history.csv', index=False)
        pd.DataFrame(all_per_dataset).to_csv(REPORT_DIR / 'stronger_dual_per_dataset.csv', index=False)
        display(pd.DataFrame(all_metrics).sort_values('macro_f1', ascending=False).head(10))

metrics_df = pd.DataFrame(all_metrics)
display(metrics_df.sort_values('macro_f1', ascending=False))


## 11. Visualize and Compare


In [ ]:
if len(all_metrics):
    metrics_df = pd.DataFrame(all_metrics)
    plt.figure(figsize=(11, 5))
    sns.barplot(data=metrics_df, x='run_name', y='macro_f1', hue='variant')
    plt.xticks(rotation=35, ha='right')
    plt.title('Stronger dual-view variants - test macro-F1')
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / 'stronger_dual_macro_f1_by_run_variant.png', dpi=160)
    plt.show()

    best = metrics_df.sort_values('macro_f1', ascending=False).iloc[0]
    print('Best model by test macro-F1:')
    display(best.to_frame().T)

    old_paths = [
        Path.cwd() / '05_Main_Model' / 'outputs' / '1D_CNN' / 'reports' / 'cnn1d_best_models_summary.csv',
        Path.cwd().parent / '05_Main_Model' / 'outputs' / '1D_CNN' / 'reports' / 'cnn1d_best_models_summary.csv',
        Path('D:/UTE/Speech Programming/Speech Project/05_Main_Model/outputs/1D_CNN/reports/cnn1d_best_models_summary.csv'),
    ]
    for p in old_paths:
        if p.exists():
            old = pd.read_csv(p)
            print('Found old 1D-CNN comparison:', p)
            display(old[['scenario', 'run_name', 'feature_name', 'accuracy', 'macro_f1', 'best_val_macro_f1']])
            break


## 12. Package Output Zip


In [ ]:
summary = {
    'notebook': '05_Dual_View_Stronger_SER',
    'model_variants': MODEL_VARIANTS,
    'output_dir': str(OUTPUT_DIR),
    'metrics_csv': str(REPORT_DIR / 'stronger_dual_metrics.csv'),
    'notes': 'Use validation macro-F1 for model selection; compare random, strict and single-dataset protocols separately.'
}
with open(REPORT_DIR / 'stronger_dual_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

zip_base = PROJECT_ROOT / '05_Dual_View_Stronger_outputs_package'
zip_path = shutil.make_archive(str(zip_base), 'zip', OUTPUT_DIR)
print('ZIP:', zip_path)
